In [38]:
import json
import pandas as pd
from openpyxl.styles import PatternFill

def process_reviews(train_filepath, test_filepath, output_filepath):
    # Load the JSON data
    with open(train_filepath, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    
    with open(test_filepath, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    # Extract train reviews into a set for fast lookup
    train_reviews = {item.get('reviews') for item in train_data if 'reviews' in item}
    
    # Prepare the output data
    output_rows = []
    for item in test_data:
        test_review = item.get('reviews', '')
        
        # Check if the test value exists in the train values
        if test_review in train_reviews:
            output_rows.append({'review': test_review, 'Flag': 'false'})
        else:
            output_rows.append({'review': test_review, 'Flag': ''})

    # Create a pandas DataFrame
    df = pd.DataFrame(output_rows)

    # Initialize the Excel writer using openpyxl
    writer = pd.ExcelWriter(output_filepath, engine='openpyxl')
    df.to_excel(writer, index=False, sheet_name='Matched_Reviews')

    # Access the workbook and worksheet to apply styling
    workbook = writer.book
    worksheet = writer.sheets['Matched_Reviews']

    # Define the red fill style
    red_fill = PatternFill(start_color='FFFF0000', end_color='FFFF0000', fill_type='solid')

    # Iterate through the rows and apply the red highlight if Flag is "false"
    # Note: openpyxl is 1-indexed, and row 1 contains the headers
    for row_idx, row_data in enumerate(output_rows, start=2):
        if row_data['Flag'] == 'false':
            # Highlight Column A (the 'review' column)
            worksheet.cell(row=row_idx, column=1).fill = red_fill

    # Save the file
    writer.close()
    print(f"Data successfully processed and saved to {output_filepath}")

# Execute the function
process_reviews('train.json', 'test.json', 'review_comparison.xlsx')

Data successfully processed and saved to review_comparison.xlsx


In [39]:
def clean_test_predictions(comparison_file, test_file, output_file):
    # 1. Load both files
    df_comparison = pd.read_excel(comparison_file)
    df_test = pd.read_csv(test_file)
    
    # 2. Normalize the 'Flag' column to string to handle Excel's automatic boolean conversion
    df_comparison['Flag'] = df_comparison['Flag'].astype(str).str.strip().str.lower()
    
    # 3. Extract the set of reviews flagged as 'false'
    flagged_reviews = set(df_comparison[df_comparison['Flag'] == 'false']['review'].dropna())
    
    # 4. Filter the test_predictions dataframe
    # The ~ operator negates the condition, keeping rows where the review is NOT in the flagged set
    df_test_cleaned = df_test[~df_test['reviews'].isin(flagged_reviews)]
    
    # 5. Save the updated dataframe to a new CSV file without the index
    df_test_cleaned.to_csv(output_file, index=False)
    
    # Print a summary of the changes
    original_count = len(df_test)
    cleaned_count = len(df_test_cleaned)
    print(f"Original row count: {original_count}")
    print(f"Removed {original_count - cleaned_count} flagged row(s).")
    print(f"Cleaned row count: {cleaned_count}")

# Execute the function referencing the files verbatim
clean_test_predictions(
    comparison_file="review_comparison.xlsx", 
    test_file="test_predictions_flags.csv", 
    output_file="cleaned_test_predictions.csv"
)

Original row count: 1851
Removed 101 flagged row(s).
Cleaned row count: 1750


In [40]:
import pandas as pd

# File paths
test_preds_file = 'test_predictions.csv'
sub_template_file = 'submission_template.csv'
review_comp_file = 'review_comparison.xlsx'
unmatched_out_file = 'remaining_unmatched_reviews.xlsx'

# 1. Load the CSV files
test_preds = pd.read_csv(test_preds_file)
sub_template = pd.read_csv(sub_template_file)

# Drop existing 'predictions' column in submission template so it can be replaced with the matched scores
if 'predictions' in sub_template.columns:
    sub_template = sub_template.drop(columns=['predictions'])

# Create a copy of the 'reviews' column in test_preds so it transfers over as a separate third column
test_preds_to_merge = test_preds[['reviews', 'predictions']].copy()
test_preds_to_merge['matched_reviews'] = test_preds_to_merge['reviews']

# 2. Merge on 'reviews' (left join keeps all original rows from submission_template)
# Resulting column order: ['reviews', 'predictions', 'matched_reviews']
updated_template = sub_template.merge(
    test_preds_to_merge[['reviews', 'predictions', 'matched_reviews']],
    on='reviews',
    how='left'
)

# Save the updated submission template
updated_template.to_csv(sub_template_file, index=False)
print(f"Updated {sub_template_file} with matched predictions (col 2) and matched reviews (col 3).")

# 3. Identify left over (unmatched) reviews where 'matched_reviews' is null
unmatched_reviews = updated_template[updated_template['matched_reviews'].isna()][['reviews']]

# 4. Compare remaining unmatched reviews with review_comparison.xlsx
if not unmatched_reviews.empty:
    try:
        review_comp = pd.read_excel(review_comp_file)
        
        if 'reviews' in review_comp.columns:
            # Keep only reviews that are STILL not present in review_comparison.xlsx
            still_unmatched = unmatched_reviews[~unmatched_reviews['reviews'].isin(review_comp['reviews'])]
        else:
            print(f"Warning: 'reviews' column not found in {review_comp_file}. Skipping comparison.")
            still_unmatched = unmatched_reviews
            
    except FileNotFoundError:
        print(f"Warning: {review_comp_file} not found. Skipping comparison step.")
        still_unmatched = unmatched_reviews

    # 5. Write any final remaining unmatched reviews to remaining_unmatched_reviews.xlsx
    if not still_unmatched.empty:
        still_unmatched.to_excel(unmatched_out_file, index=False)
        print(f"Saved {len(still_unmatched)} remaining unmatched reviews to {unmatched_out_file}.")
    else:
        print("All leftover reviews were found in the comparison file.")
else:
    print("All reviews matched directly from test predictions. No leftovers!")

Updated submission_template.csv with matched predictions (col 2) and matched reviews (col 3).
All reviews matched directly from test predictions. No leftovers!


In [41]:
# 1. Load the files
sub_df = pd.read_csv('submission_template.csv', low_memory=False)
pred_df = pd.read_csv('test_predictions.csv', low_memory=False)
comp_df = pd.read_excel('review_comparison.xlsx')

# Remove completely blank trailing rows (common when exporting snippets from Excel)
sub_df = sub_df.dropna(how='all').copy()
pred_df = pred_df.dropna(how='all').copy()
comp_df = comp_df.dropna(how='all').copy()

# 2. Match 'reviews' between the two CSVs and copy 'predictions' into submission_template
# Create a lookup dictionary/Series mapping each unique review to its prediction
pred_map = (
    pred_df.dropna(subset=['reviews'])
    .drop_duplicates(subset=['reviews'])
    .set_index('reviews')['predictions']
)

# Update 'predictions' in submission_template only where a match exists
matched_mask = sub_df['reviews'].isin(pred_map.index)
sub_df.loc[matched_mask, 'predictions'] = sub_df.loc[matched_mask, 'reviews'].map(pred_map)

# Save the updated submission file
sub_df.to_csv('updated_submission_template.csv', index=False)

# 3. Find leftover rows in test_predictions that did not match submission_template
sub_reviews = set(sub_df['reviews'].dropna())
leftover_pred_df = pred_df[~pred_df['reviews'].isin(sub_reviews)].copy()

# 4. Match leftover rows against the 'review' column in review_comparison.xlsx and drop matches
comp_reviews = set(comp_df['review'].dropna())
final_leftover_df = leftover_pred_df[~leftover_pred_df['reviews'].isin(comp_reviews)].copy()

# 5. Save any remaining unmatched rows to an Excel file
final_leftover_df.to_excel('leftover_predictions.xlsx', index=False)

print(f"Submission rows updated: {matched_mask.sum()}")
print(f"Leftover rows after CSV comparison: {len(leftover_pred_df)}")
print(f"Final leftover rows saved to Excel: {len(final_leftover_df)}")

Submission rows updated: 1851
Leftover rows after CSV comparison: 0
Final leftover rows saved to Excel: 0
